In [30]:
import pandas as pd
import numpy as np
from scipy import stats

In [59]:
data = pd.read_csv('C:/Users/Admin/Downloads/datasets/credit_card_transactions.csv')
data.head().T

,0,1,2,3,4
Unnamed: 0,0,1,2,3,4
trans_date_trans_time,2019-01-01 00:00:18,2019-01-01 00:00:44,2019-01-01 00:00:51,2019-01-01 00:01:16,2019-01-01 00:03:06
cc_num,2703186189652095,630423337322,38859492057661,3534093764340240,375534208663984
merchant,"fraud_Rippin, Kub and Mann","fraud_Heller, Gutmann and Zieme",fraud_Lind-Buckridge,"fraud_Kutch, Hermiston and Farrell",fraud_Keeling-Crist
category,misc_net,grocery_pos,entertainment,gas_transport,misc_pos
amt,4.97,107.23,220.11,45.0,41.96
first,Jennifer,Stephanie,Edward,Jeremy,Tyler
last,Banks,Gill,Sanchez,White,Garcia
gender,F,F,M,M,M
street,561 Perry Cove,43039 Riley Greens Suite 393,594 White Dale Suite 530,9443 Cynthia Court Apt. 038,408 Bradley Rest


```
Let's create features called age_at_trans, hour_of_trans, day_of_week, and is_weekend based on transaction time and DOB!

In [61]:
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])
data['dob'] = pd.to_datetime(data['dob'])

data['age_at_trans'] = data['trans_date_trans_time'].dt.year - data['dob'].dt.year
data['hour'] = data['trans_date_trans_time'].dt.hour
data['day_of_week'] = data['trans_date_trans_time'].dt.dayofweek
data['is_weekend'] = data['day_of_week'] >= 5
data[['age_at_trans', 'hour', 'day_of_week', 'is_weekend']]

,age_at_trans,hour,day_of_week,is_weekend
0,31,0,1,False
1,41,0,1,False
2,57,0,1,False
3,52,0,1,False
4,33,0,1,False
...,...,...,...,...
1296670,59,12,6,True
1296671,41,12,6,True
1296672,53,12,6,True
1296673,40,12,6,True


```
Counting Unique Rows!

In [62]:
data.nunique()

Unnamed: 0               1296675
trans_date_trans_time    1274791
cc_num                       983
merchant                     693
category                      14
amt                        52928
first                        352
last                         481
gender                         2
street                       983
city                         894
state                         51
zip                          970
lat                          968
long                         969
city_pop                     879
job                          494
dob                          968
trans_num                1296675
unix_time                1274823
merch_lat                1247805
merch_long               1275745
is_fraud                       2
merch_zipcode              28336
age_at_trans                  83
hour                          24
day_of_week                    7
is_weekend                     2
dtype: int64

```
Not all features are necessary or even useful for EDA and Modeling.

In [63]:
data = data.drop(labels=['Unnamed: 0', 'trans_date_trans_time', 'zip', 'first', 'last', 
                'cc_num', 'street', 'trans_num', 'unix_time', 'merchant', 
                'merch_zipcode', 'job', 'dob', 'city', 'city_pop'], axis=1)

```
Let's check if there's any duplicated data!

In [64]:
data.duplicated().sum()

np.int64(0)

In [65]:
data.isnull().sum()

category        0
amt             0
gender          0
state           0
lat             0
long            0
merch_lat       0
merch_long      0
is_fraud        0
age_at_trans    0
hour            0
day_of_week     0
is_weekend      0
dtype: int64

```
No duplicated or NaN rows exists, a good sign!

```
Let's summarize the dataset.

In [66]:
data.head()

,category,amt,gender,state,lat,long,merch_lat,merch_long,is_fraud,age_at_trans,hour,day_of_week,is_weekend
0,misc_net,4.97,F,NC,36.0788,-81.1781,36.011293,-82.048315,0,31,0,1,False
1,grocery_pos,107.23,F,WA,48.8878,-118.2105,49.159047,-118.186462,0,41,0,1,False
2,entertainment,220.11,M,ID,42.1808,-112.2620,43.150704,-112.154481,0,57,0,1,False
3,gas_transport,45.00,M,MT,46.2306,-112.1138,47.034331,-112.561071,0,52,0,1,False
4,misc_pos,41.96,M,VA,38.4207,-79.4629,38.674999,-78.632459,0,33,0,1,False


```
Range can be useful, so let's also see that!

In [72]:
num_cols = data.select_dtypes(['int64', 'float64']).drop('is_fraud', axis=1).columns.tolist()

for col in num_cols:
    max_ = np.max(data[col])
    min_ = np.min(data[col])
    range_ = max_ - min_
    print(f'\nRange of {col}: {range_}')


Range of amt: 28947.9

Range of lat: 46.66619999999999

Range of long: 97.72200000000001

Range of merch_lat: 48.482482

Range of merch_long: 99.72034000000001


...

### Let's calculate the Correlation between Features ###

In [69]:
corr = data.corr(numeric_only=True)
corr.T

,amt,lat,long,merch_lat,merch_long,is_fraud,age_at_trans,hour,day_of_week,is_weekend
amt,1.000000,-0.001926,-0.000187,-0.001873,-0.000151,0.219404,-0.009724,-0.022811,-0.001001,-0.002054
lat,-0.001926,1.000000,-0.015533,0.993592,-0.015509,0.001894,0.047868,-0.011508,0.000498,-0.000022
long,-0.000187,-0.015533,1.000000,-0.015452,0.999120,0.001721,-0.030220,-0.002290,0.001593,0.001956
merch_lat,-0.001873,0.993592,-0.015452,1.000000,-0.015431,0.001741,0.047480,-0.011378,0.000263,-0.000210
merch_long,-0.000151,-0.015509,0.999120,-0.015431,1.000000,0.001721,-0.030142,-0.002325,0.001553,0.001918
is_fraud,0.219404,0.001894,0.001721,0.001741,0.001721,1.000000,0.012453,0.013799,0.001739,-0.003644
age_at_trans,-0.009724,0.047868,-0.030220,0.047480,-0.030142,0.012453,1.000000,-0.172847,-0.012908,-0.013838
hour,-0.022811,-0.011508,-0.002290,-0.011378,-0.002325,0.013799,-0.172847,1.000000,0.000219,-0.000079
day_of_week,-0.001001,0.000498,0.001593,0.000263,0.001553,0.001739,-0.012908,0.000219,1.000000,0.826107
is_weekend,-0.002054,-0.000022,0.001956,-0.000210,0.001918,-0.003644,-0.013838,-0.000079,0.826107,1.000000


### Let's Summarize the Dataset! ###

In [67]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
amt,1296675.0,70.351035,160.316039,1.000000,9.650000,47.520000,83.140000,28948.900000
lat,1296675.0,38.537622,5.075808,20.027100,34.620500,39.354300,41.940400,66.693300
long,1296675.0,-90.226335,13.759077,-165.672300,-96.798000,-87.476900,-80.158000,-67.950300
merch_lat,1296675.0,38.537338,5.109788,19.027785,34.733572,39.365680,41.957164,67.510267
merch_long,1296675.0,-90.226465,13.771091,-166.671242,-96.897276,-87.438392,-80.236796,-66.950902
is_fraud,1296675.0,0.005789,0.075863,0.000000,0.000000,0.000000,0.000000,1.000000
age_at_trans,1296675.0,46.029298,17.382373,14.000000,33.000000,44.000000,57.000000,96.000000
hour,1296675.0,12.804858,6.817824,0.000000,7.000000,14.000000,19.000000,23.000000
day_of_week,1296675.0,3.070604,2.198153,0.000000,1.000000,3.000000,5.000000,6.000000


# amt: 
Huge max (28948.9) → clear outliers, likely very large transactions

Median (50%) = 9.65 → most transactions are small

lat / long and merch_lat / merch_long

Lat/long ranges are wide (maybe anonymized or synthetic)

Can be used for distance calculation between customer and merchant

# is_fraud: 
Very imbalanced: ~0.58% fraud

# age_at_trans :
Ages mostly between 33–57, min 14, max 96

# hour / day_of_week:
Transactions happen throughout the day/week

Useful for temporal pattern features